In [1]:
import requests
import pandas as pd

# Consumir la API
response = requests.get('https://randomuser.me/api/?results=100')
data = response.json()

# Verificar que llegaron datos
print('Status code:', response.status_code)
print('Total registros:', len(data['results']))

Status code: 200
Total registros: 100


In [2]:
clientes = []

for i, persona in enumerate(data['results']):
    clientes.append({
        'cliente_id': i + 1,
        'nombre': persona['name']['first'] + ' ' + persona['name']['last'],
        'ciudad': persona['location']['city'],
        'pais': persona['location']['country']
    })

df_clientes = pd.DataFrame(clientes)

print('Tabla de clientes:')
print(df_clientes.shape)
df_clientes.head()

Tabla de clientes:
(100, 4)


,cliente_id,nombre,ciudad,pais
0,1,Patrizia Barbier,Hausen (Ag),Switzerland
1,2,اميرحسين کریمی,سبزوار,Iran
2,3,Mathilde Petersen,Viby J.,Denmark
3,4,Candice Howard,Bellevue,United States
4,5,Ataíde Souza,Divinópolis,Brazil


In [12]:
response_paises = requests.get(
    'https://restcountries.com/v3.1/all',
    params={'fields': 'name,region,subregion,capital'}
)
data_paises = response_paises.json()

print('Status code:', response_paises.status_code)
print('Total países:', len(data_paises))
print('Primer elemento:', data_paises[0])

Status code: 200
Total países: 250
Primer elemento: {'name': {'common': 'Ivory Coast', 'official': "Republic of Côte d'Ivoire", 'nativeName': {'fra': {'official': "République de Côte d'Ivoire", 'common': "Côte d'Ivoire"}}}, 'capital': ['Yamoussoukro'], 'region': 'Africa', 'subregion': 'Western Africa'}


In [13]:
regiones = []

for pais in data_paises:
    # Verificar que el elemento sea un diccionario antes de procesarlo
    if not isinstance(pais, dict):
        continue
    
    regiones.append({
        'pais': pais.get('name', {}).get('common', '') if isinstance(pais.get('name'), dict) else '',
        'continente': pais.get('region', ''),
        'subregion': pais.get('subregion', ''),
        'capital': pais.get('capital', [''])[0] if pais.get('capital') else ''
    })

df_regiones = pd.DataFrame(regiones)

print('Tabla de regiones:')
print(df_regiones.shape)
df_regiones.head()

Tabla de regiones:
(250, 4)


,pais,continente,subregion,capital
0,Ivory Coast,Africa,Western Africa,Yamoussoukro
1,Italy,Europe,Southern Europe,Rome
2,Kyrgyzstan,Asia,Central Asia,Bishkek
3,Papua New Guinea,Oceania,Melanesia,Port Moresby
4,Fiji,Oceania,Melanesia,Suva


In [14]:
# Países en el dataset de ventas
paises_ventas = [
    'USA', 'France', 'Norway', 'Australia', 'Finland',
    'Austria', 'UK', 'Spain', 'Sweden', 'Singapore',
    'Canada', 'Japan', 'Italy', 'Denmark', 'Belgium',
    'Philippines', 'Germany', 'Switzerland', 'Ireland'
]

# Buscar cuáles coinciden
coincidencias = df_regiones[df_regiones['pais'].isin(paises_ventas)]
print('Países que coinciden:', len(coincidencias))
print('Países que NO coinciden:', 19 - len(coincidencias))
print('\nPaíses encontrados:')
print(coincidencias['pais'].tolist())

Países que coinciden: 17
Países que NO coinciden: 2

Países encontrados:
['Italy', 'Finland', 'Japan', 'Belgium', 'France', 'Ireland', 'Norway', 'Spain', 'Australia', 'Switzerland', 'Philippines', 'Sweden', 'Germany', 'Denmark', 'Austria', 'Canada', 'Singapore']


In [15]:
# Buscar USA y UK en la tabla de regiones
buscar = ['United States', 'United Kingdom', 'USA', 'UK']
print(df_regiones[df_regiones['pais'].isin(buscar)])

               pais continente        subregion           capital
86   United Kingdom     Europe  Northern Europe            London
148   United States   Americas    North America  Washington, D.C.


In [16]:
# Mapeo de nombres del CSV al nombre real en la API
mapeo_paises = {
    'USA': 'United States',
    'UK': 'United Kingdom'
}

# Cargar ventas y aplicar el mapeo
df_ventas = pd.read_csv('../4_output/ventas_limpio.csv')
df_ventas['country'] = df_ventas['country'].replace(mapeo_paises)

print('Países únicos después del mapeo:')
print(df_ventas['country'].unique())

Países únicos después del mapeo:
<StringArray>
[ 'United States',         'France',         'Norway',      'Australia',
        'Finland',        'Austria', 'United Kingdom',          'Spain',
         'Sweden',      'Singapore',         'Canada',          'Japan',
          'Italy',        'Denmark',        'Belgium',    'Philippines',
        'Germany',    'Switzerland',        'Ireland']
Length: 19, dtype: str


In [17]:
import numpy as np

# Fijar semilla para que el resultado sea siempre igual
np.random.seed(42)

# Asignar un cliente_id aleatorio a cada venta
df_ventas['cliente_id'] = np.random.randint(1, 101, size=len(df_ventas))

print('cliente_id asignado correctamente')
print(df_ventas[['ordernumber', 'country', 'cliente_id']].head(10))

cliente_id asignado correctamente
   ordernumber        country  cliente_id
0        10107  United States          52
1        10121         France          93
2        10134         France          15
3        10145  United States          72
4        10159  United States          61
5        10168  United States          21
6        10180         France          83
7        10188         Norway          87
8        10201  United States          75
9        10211         France          75


In [18]:
df_ventas = df_ventas.merge(
    df_regiones,
    left_on='country',
    right_on='pais',
    how='left'
)

# Eliminar columna duplicada
df_ventas = df_ventas.drop(columns=['pais'])

print('Columnas finales de ventas:')
print(df_ventas.columns.tolist())
df_ventas.head()

Columnas finales de ventas:
['ordernumber', 'orderdate', 'quantityordered', 'priceeach', 'sales', 'country', 'customername', 'productline', 'dealsize', 'status', 'producto_id', 'cliente_id', 'continente', 'subregion', 'capital']


,ordernumber,orderdate,quantityordered,priceeach,sales,country,customername,productline,dealsize,status,producto_id,cliente_id,continente,subregion,capital
0,10107,2003-02-24,30,95.70,2871.00,United States,Land of Toys Inc.,Motorcycles,Small,Shipped,1,52,Americas,North America,"Washington, D.C."
1,10121,2003-05-07,34,81.35,2765.90,France,Reims Collectables,Motorcycles,Small,Shipped,1,93,Europe,Western Europe,Paris
2,10134,2003-07-01,41,94.74,3884.34,France,Lyon Souveniers,Motorcycles,Medium,Shipped,1,15,Europe,Western Europe,Paris
3,10145,2003-08-25,45,83.26,3746.70,United States,Toys4GrownUps.com,Motorcycles,Medium,Shipped,1,72,Americas,North America,"Washington, D.C."
4,10159,2003-10-10,49,100.00,5205.27,United States,Corporate Gift Ideas Co.,Motorcycles,Medium,Shipped,1,61,Americas,North America,"Washington, D.C."


In [19]:
print('Nulos en continente:', df_ventas['continente'].isnull().sum())
print('Continentes únicos:', df_ventas['continente'].unique())

Nulos en continente: 0
Continentes únicos: <StringArray>
['Americas', 'Europe', 'Oceania', 'Asia']
Length: 4, dtype: str


In [20]:
# Guardar ventas completo
df_ventas.to_csv('../4_output/ventas_limpio.csv', index=False)

# Guardar clientes
df_clientes.to_csv('../4_output/clientes.csv', index=False)

# Guardar regiones
df_regiones.to_csv('../4_output/regiones.csv', index=False)

print('Archivos guardados en 4_output/ ✅')
print('\nResumen:')
print(f'  ventas_limpio.csv  → {len(df_ventas)} filas')
print(f'  clientes.csv       → {len(df_clientes)} filas')
print(f'  regiones.csv       → {len(df_regiones)} filas')

Archivos guardados en 4_output/ ✅

Resumen:
  ventas_limpio.csv  → 2823 filas
  clientes.csv       → 100 filas
  regiones.csv       → 250 filas
